# SeatGuard — Station 2: Stitch Defect YOLO Training

Trains YOLOv8 on the Roboflow **Sewing Defects** dataset (3,748 images, CC BY 4.0)  
and exports `yolo_defect.onnx` for `models/station_2/`.

**Runtime:** GPU → Runtime → Change runtime type → T4 GPU

**Classes trained:**
- `broken_thread`
- `skip_stitch`

**Expected output:** `yolo_defect.onnx` (download at the end)

## Step 1 — Install dependencies

In [ ]:
!pip install ultralytics roboflow --quiet

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 2 — Download dataset from Roboflow

Go to https://universe.roboflow.com/sewing/sewing-defects  
Click **Download** → **YOLOv8** format → **Show download code**  
Copy your API key and paste it below.

In [ ]:
from roboflow import Roboflow

# ── Paste your Roboflow API key here ──────────────────────────────────────────
ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"   # <── replace this
# ─────────────────────────────────────────────────────────────────────────────

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("sewing").project("sewing-defects")
dataset = project.version(1).download("yolov8")

print(f"Dataset location: {dataset.location}")
print(f"Classes: {dataset.classes}")

## Step 3 — Verify dataset structure

Should show: `train/`, `valid/`, `test/` folders and a `data.yaml`

In [ ]:
import os, yaml

data_yaml_path = os.path.join(dataset.location, "data.yaml")

with open(data_yaml_path) as f:
    data_cfg = yaml.safe_load(f)

print("data.yaml contents:")
print(yaml.dump(data_cfg, default_flow_style=False))

# Count images per split
for split in ["train", "valid", "test"]:
    img_dir = os.path.join(dataset.location, split, "images")
    if os.path.exists(img_dir):
        n = len(os.listdir(img_dir))
        print(f"{split}: {n} images")

## Step 4 — Patch data.yaml class names to match SeatGuard

The app expects these exact class names:
- `broken_thread`  
- `skip_stitch`

If the Roboflow dataset uses different names (e.g. `broken-thread`, `skip stitch`),  
this cell remaps them.

In [ ]:
# Map from whatever names Roboflow uses → SeatGuard canonical names
# Edit the LEFT side if your dataset uses different names
NAME_MAP = {
    "broken-thread":   "broken_thread",
    "broken thread":   "broken_thread",
    "skip-stitch":     "skip_stitch",
    "skip stitch":     "skip_stitch",
    "skipped stitch":  "skip_stitch",
    # already correct → no-op
    "broken_thread":   "broken_thread",
    "skip_stitch":     "skip_stitch",
}

original_names = data_cfg.get("names", [])
patched_names  = [NAME_MAP.get(n, n) for n in original_names]

print(f"Original class names : {original_names}")
print(f"Patched  class names : {patched_names}")

data_cfg["names"] = patched_names
with open(data_yaml_path, "w") as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print("\ndata.yaml updated.")

## Step 5 — Train YOLOv8

- Model: `yolov8s` (small — good balance of speed and accuracy for ONNX inference)
- Image size: `640` (standard; ROI crops are resized to this)
- Epochs: `50` (enough for this dataset size; increase to 100 if mAP plateaus)
- Threshold used at inference: `0.50` (matches `station_2_stitch.yaml`)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")   # pretrained on COCO — fine-tunes faster

results = model.train(
    data=data_yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,           # reduce to 8 if you get OOM on T4
    lr0=0.01,
    patience=10,        # early stopping: stop if no improvement for 10 epochs
    augment=True,       # mosaic, flip, HSV jitter — helps generalise to real seat images
    degrees=5.0,        # small rotation — seams are roughly horizontal
    fliplr=0.5,
    flipud=0.0,         # seams don't appear upside down
    hsv_h=0.015,        # thread hue can vary between variants
    hsv_s=0.4,
    hsv_v=0.3,
    project="seatguard_station2",
    name="yolo_defect",
    exist_ok=True,
    device=0,           # GPU 0
    conf=0.50,          # confidence threshold matching station config
    verbose=True,
)

print("\nTraining complete.")
print(f"Best weights: {results.save_dir}/weights/best.pt")

## Step 6 — Evaluate on validation set

In [ ]:
best_pt = "seatguard_station2/yolo_defect/weights/best.pt"
trained_model = YOLO(best_pt)

metrics = trained_model.val(data=data_yaml_path, conf=0.50)

print(f"\nmAP50      : {metrics.box.map50:.4f}")
print(f"mAP50-95   : {metrics.box.map:.4f}")
print(f"Precision  : {metrics.box.mp:.4f}")
print(f"Recall     : {metrics.box.mr:.4f}")

# Per-class breakdown
print("\nPer-class AP50:")
for cls_name, ap in zip(trained_model.names.values(), metrics.box.ap50):
    print(f"  {cls_name:20s}: {ap:.4f}")

## Step 7 — Export to ONNX

Exports with **dynamic batch size** and **opset 17** — compatible with OpenCV DNN  
and the ONNX Runtime backend used by the SeatGuard app.

In [ ]:
trained_model.export(
    format="onnx",
    imgsz=640,
    opset=17,
    dynamic=True,       # variable batch size — lets the app pass 1 image at a time
    simplify=True,      # onnx-simplifier: removes dead nodes, smaller file
)

onnx_path = best_pt.replace(".pt", ".onnx")
print(f"ONNX model saved to: {onnx_path}")

# Verify it loads correctly
import onnx
m = onnx.load(onnx_path)
onnx.checker.check_model(m)
print("ONNX model check passed.")

# Show input/output shape
for inp in m.graph.input:
    shape = [d.dim_value for d in inp.type.tensor_type.shape.dim]
    print(f"Input  '{inp.name}': {shape}")
for out in m.graph.output:
    shape = [d.dim_value for d in out.type.tensor_type.shape.dim]
    print(f"Output '{out.name}': {shape}")

## Step 8 — Quick smoke test on a sample image

In [ ]:
import glob, random
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Pick a random validation image
val_images = glob.glob(os.path.join(dataset.location, "valid", "images", "*"))
sample_img = random.choice(val_images)
print(f"Testing on: {sample_img}")

preds = trained_model.predict(sample_img, conf=0.50, verbose=False)
result = preds[0]

img = Image.open(sample_img)
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
ax.imshow(img)

colors = {"broken_thread": "red", "skip_stitch": "orange"}
for box in result.boxes:
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    cls_id = int(box.cls[0])
    cls_name = trained_model.names[cls_id]
    conf = float(box.conf[0])
    color = colors.get(cls_name, "blue")
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                               linewidth=2, edgecolor=color, facecolor="none")
    ax.add_patch(rect)
    ax.text(x1, y1-5, f"{cls_name} {conf:.2f}",
            color=color, fontsize=9, weight="bold")

ax.set_title(f"Detections: {len(result.boxes)}")
ax.axis("off")
plt.tight_layout()
plt.show()

## Step 9 — Download the ONNX file

This downloads `yolo_defect.onnx` to your computer.  
Place it at `models/station_2/yolo_defect.onnx` in the SeatGuard project.

In [ ]:
import shutil
from google.colab import files

# Copy to a clean name for download
output_name = "yolo_defect.onnx"
shutil.copy(onnx_path, output_name)

print(f"File size: {os.path.getsize(output_name) / 1e6:.1f} MB")
files.download(output_name)
print("\nDownload started.")
print("Place this file at:  models/station_2/yolo_defect.onnx")

## Notes

### What to do after downloading
1. Copy `yolo_defect.onnx` to `models/station_2/yolo_defect.onnx`
2. In `configs/stations/station_2_stitch.yaml` change `backend: "mock"` → `backend: "onnx"`
3. Run the app in replay mode and check Station 2 detections

### Expected mAP targets
| Metric | Acceptable | Good |
|--------|-----------|------|
| mAP50  | > 0.55    | > 0.70 |
| Precision | > 0.60 | > 0.75 |
| Recall    | > 0.55 | > 0.70 |

If mAP50 is below 0.55 after 50 epochs:
- Increase epochs to 100
- Try `yolov8m.pt` (medium) instead of small
- Check that class names patched correctly in Step 4

### Limitations of this baseline
- Trained on garment sewing images — not automotive seat leather
- Will detect broken thread and skip stitches well
- Thread color deviation and seam position are handled by classical OpenCV (no model needed)
- Fine-tune with 200+ client seat images after shadow mode collection for production accuracy

### Class ID mapping (important for inference)
After training, verify class IDs match what the app expects:
```
0: broken_thread
1: skip_stitch
```
The order depends on alphabetical sort by Roboflow. Check `trained_model.names` output.